# 07 — Baseline Model

## Objective

This notebook establishes transparent baseline models for predicting
observed poultry disease outbreak counts in Nigeria.

The modelling target is:

`target_outbreak_count`

Only WAHIS records with an observed outbreak count are eligible for
supervised modelling.

## Modelling Scenario

The initial modelling scenario is retrospective/end-of-semester
prediction.

The climate variables describe the same ADM1 × disease × semester
period as the outbreak target. Therefore, the resulting models should
not yet be interpreted as pre-semester early-warning forecasts.

## Baseline Strategy

Before applying machine-learning models, we establish simple
historical baselines:

1. Previous-semester persistence
2. Previous 3-semester historical mean
3. Historical 3-semester maximum

These baselines provide transparent benchmarks against which more
complex models can be evaluated.

## Important Surveillance Principle

Unavailable WAHIS outbreak counts are not treated as zero.

Only records with observed outbreak counts are used as supervised
targets.

## Validation Strategy

Because the data are temporal, random train/test splitting is avoided.

Model evaluation will use chronological validation so that future
observations are never used to predict earlier observations.

In [1]:
# ======================================================================
# CELL 2 — LOAD THE FEATURE-ENGINEERED DATASET
# ======================================================================
# Purpose:
#   Load the completed feature-engineering dataset from Notebook 06
#   and prepare the observed outbreak records for baseline evaluation.
#
# Important:
#   - Unavailable outbreak observations remain excluded from supervised
#     target evaluation.
#   - No random train/test split is used.
#   - The temporal structure of the data is preserved.
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------
# 1. Define project paths
# ----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_climate_integrated.csv"
)


# ----------------------------------------------------------------------
# 2. Load the integrated dataset
# ----------------------------------------------------------------------
# Notebook 06 feature engineering was performed in memory, so we first
# verify that the required source columns are available in the saved
# integrated dataset.

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("BASELINE MODEL — DATA LOADING")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")


# ----------------------------------------------------------------------
# 3. Reconstruct the observed target
# ----------------------------------------------------------------------
# Only observed outbreak counts are valid supervised targets.
#
# Unavailable outbreak observations remain NaN.

df["target_outbreak_count"] = df["new_outbreaks"].where(
    df["outbreak_observation_status"] == "Observed"
)

df["target_observed"] = df["target_outbreak_count"].notna()


# ----------------------------------------------------------------------
# 4. Extract semester order
# ----------------------------------------------------------------------
# The original semester field contains values such as:
#   "Jan-Jun 2021"
#   "Jul-Dec 2021"
#
# We extract the semester component and assign chronological order.

df["semester_name"] = df["semester"].str.extract(
    r"^(Jan-Jun|Jul-Dec)"
)[0]

semester_order_map = {
    "Jan-Jun": 1,
    "Jul-Dec": 2,
}

df["semester_order"] = df["semester_name"].map(
    semester_order_map
)


# ----------------------------------------------------------------------
# 5. Create a chronological temporal index
# ----------------------------------------------------------------------

df["temporal_order"] = (
    (df["year"] - df["year"].min()) * 2
    + df["semester_order"]
)


# ----------------------------------------------------------------------
# 6. Keep only supervised observations
# ----------------------------------------------------------------------
# This does NOT interpret unavailable observations as zero.
# It simply restricts the baseline evaluation dataset to records with
# an observed target.

modeling_df = df[
    df["target_observed"]
].copy()


# ----------------------------------------------------------------------
# 7. Validate the modelling dataset
# ----------------------------------------------------------------------

assert len(modeling_df) == 445, (
    "Unexpected number of observed outbreak records."
)

assert modeling_df["target_outbreak_count"].notna().all(), (
    "Observed modelling records contain missing targets."
)

assert (modeling_df["target_outbreak_count"] > 0).all(), (
    "Observed outbreak targets contain non-positive values."
)

assert modeling_df["temporal_order"].notna().all(), (
    "Temporal ordering contains missing values."
)


# ----------------------------------------------------------------------
# 8. Sort chronologically
# ----------------------------------------------------------------------

modeling_df = modeling_df.sort_values(
    [
        "temporal_order",
        "adm1_key",
        "disease",
        "analytical_record_id",
    ]
).reset_index(drop=True)


# ----------------------------------------------------------------------
# 9. Display data summary
# ----------------------------------------------------------------------

print("\nObserved modelling records:")
print(f"  {len(modeling_df)}")

print("\nTemporal coverage:")
print(
    f"  {modeling_df['year'].min()}–"
    f"{modeling_df['year'].max()}"
)

print("\nTarget summary:")
display(
    modeling_df[
        "target_outbreak_count"
    ].describe().to_frame("target_outbreak_count")
)

print("\nObserved records by year:")
display(
    modeling_df.groupby("year")
    .size()
    .rename("observed_records")
    .reset_index()
)

print("\n" + "=" * 70)
print("BASELINE MODEL DATA LOADING VALIDATION PASSED")
print("=" * 70)

BASELINE MODEL — DATA LOADING

Dataset shape: (1180, 39)

Observed modelling records:
  445

Temporal coverage:
  2020–2025

Target summary:


,target_outbreak_count
count,445.000000
mean,6.840449
std,12.872353
min,1.000000
25%,1.000000
50%,3.000000
75%,7.000000
max,184.000000



Observed records by year:


,year,observed_records
0,2020,31
1,2021,112
2,2022,94
3,2023,44
4,2024,68
5,2025,96



BASELINE MODEL DATA LOADING VALIDATION PASSED


In [2]:
# ============================================================
# CELL 3 — CONSTRUCT SEMESTER-LEVEL BASELINE HISTORY
# ============================================================
#
# Purpose:
# Create a unique ADM1 × disease × semester modelling table.
#
# This is important because the raw WAHIS dataset can contain
# multiple records for the same ADM1, disease, year and semester.
# Baselines should therefore be calculated at the semester level
# rather than by shifting individual WAHIS rows.
#
# Missing outbreak counts are preserved as NaN.
# They are NOT converted to zero.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create a clean semester identifier
# ------------------------------------------------------------

df["semester_name"] = (
    df["semester"]
    .astype(str)
    .str.extract(r"^(Jan-Jun|Jul-Dec)", expand=False)
)

# Validate semester extraction
assert df["semester_name"].isin(["Jan-Jun", "Jul-Dec"]).all(), (
    "Unexpected semester values detected."
)

# ------------------------------------------------------------
# 2. Assign chronological semester order
# ------------------------------------------------------------

semester_order_map = {
    "Jan-Jun": 1,
    "Jul-Dec": 2
}

df["semester_order"] = df["semester_name"].map(semester_order_map)

# Create a continuous temporal index.
# This allows semesters to be ordered across years.
df["temporal_order"] = (
    (df["year"] - df["year"].min()) * 2
    + df["semester_order"]
)

# ------------------------------------------------------------
# 3. Create the modelling outbreak count
# ------------------------------------------------------------

# Only observed WAHIS outbreak counts are used.
# Unavailable values remain NaN.
df["target_outbreak_count"] = df["new_outbreaks"].where(
    df["outbreak_observation_status"] == "Observed"
)

# ------------------------------------------------------------
# 4. Construct the unique ADM1 × disease × semester table
# ------------------------------------------------------------

history_table = (
    df[
        [
            "adm1_key",
            "disease",
            "year",
            "semester_name",
            "semester_order",
            "temporal_order",
            "target_outbreak_count"
        ]
    ]
    .groupby(
        [
            "adm1_key",
            "disease",
            "year",
            "semester_name",
            "semester_order",
            "temporal_order"
        ],
        dropna=False
    )["target_outbreak_count"]
    .sum(min_count=1)
    .reset_index()
)

# ------------------------------------------------------------
# 5. Sort the historical table chronologically
# ------------------------------------------------------------

history_table = history_table.sort_values(
    [
        "adm1_key",
        "disease",
        "temporal_order"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Create previous-semester persistence baseline
# ------------------------------------------------------------

# Shift within each ADM1 × disease series.
#
# The shift is performed BEFORE the current period is evaluated,
# ensuring that the current target cannot enter its own baseline.

history_table["baseline_previous_semester"] = (
    history_table
    .groupby(["adm1_key", "disease"])["target_outbreak_count"]
    .shift(1)
)

# ------------------------------------------------------------
# 7. Create previous 3-semester historical mean
# ------------------------------------------------------------

# First shift by one semester so the current target is excluded.
# Then calculate the rolling mean using only preceding periods.

history_table["baseline_previous_3_mean"] = (
    history_table
    .groupby(["adm1_key", "disease"])["target_outbreak_count"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

# ------------------------------------------------------------
# 8. Create previous 3-semester historical maximum
# ------------------------------------------------------------

history_table["baseline_previous_3_max"] = (
    history_table
    .groupby(["adm1_key", "disease"])["target_outbreak_count"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=3,
            min_periods=1
        ).max()
    )
)

# ------------------------------------------------------------
# 9. Add history availability indicators
# ------------------------------------------------------------

history_table["previous_semester_available"] = (
    history_table["baseline_previous_semester"].notna()
)

history_table["previous_3_mean_available"] = (
    history_table["baseline_previous_3_mean"].notna()
)

history_table["previous_3_max_available"] = (
    history_table["baseline_previous_3_max"].notna()
)

# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

# The historical table should contain one row per
# ADM1 × disease × year × semester combination.

duplicate_history_keys = history_table.duplicated(
    subset=[
        "adm1_key",
        "disease",
        "year",
        "semester_name"
    ]
).sum()

assert duplicate_history_keys == 0, (
    f"Duplicate historical keys detected: {duplicate_history_keys}"
)

# Baseline predictions must never be negative.
baseline_columns = [
    "baseline_previous_semester",
    "baseline_previous_3_mean",
    "baseline_previous_3_max"
]

for column in baseline_columns:
    assert (
        history_table[column].dropna() >= 0
    ).all(), f"Negative values detected in {column}"

# ------------------------------------------------------------
# 11. Display validation summary
# ------------------------------------------------------------

print("=" * 70)
print("SEMESTER-LEVEL BASELINE HISTORY CONSTRUCTION")
print("=" * 70)

print(f"\nUnique ADM1 × disease × semester periods: {len(history_table):,}")

print(
    "\nPeriods with observed outbreak counts:",
    history_table["target_outbreak_count"].notna().sum()
)

print(
    "Periods without observed outbreak counts:",
    history_table["target_outbreak_count"].isna().sum()
)

print("\nBaseline availability:")

for column, label in [
    (
        "baseline_previous_semester",
        "Previous semester"
    ),
    (
        "baseline_previous_3_mean",
        "Previous 3-semester mean"
    ),
    (
        "baseline_previous_3_max",
        "Previous 3-semester maximum"
    )
]:
    available = history_table[column].notna().sum()
    print(
        f"  {label}: "
        f"{available:,} available / "
        f"{len(history_table):,} total"
    )

print("\nSample baseline history:")
display(
    history_table[
        [
            "adm1_key",
            "disease",
            "year",
            "semester_name",
            "target_outbreak_count",
            "baseline_previous_semester",
            "baseline_previous_3_mean",
            "baseline_previous_3_max"
        ]
    ]
    .dropna(subset=["target_outbreak_count"])
    .head(15)
)

print("\nSEMESTER-LEVEL BASELINE HISTORY VALIDATION PASSED")

SEMESTER-LEVEL BASELINE HISTORY CONSTRUCTION

Unique ADM1 × disease × semester periods: 739

Periods with observed outbreak counts: 445
Periods without observed outbreak counts: 294

Baseline availability:
  Previous semester: 357 available / 739 total
  Previous 3-semester mean: 406 available / 739 total
  Previous 3-semester maximum: 406 available / 739 total

Sample baseline history:


,adm1_key,disease,year,semester_name,target_outbreak_count,baseline_previous_semester,baseline_previous_3_mean,baseline_previous_3_max
1,Abia,Fowl typhoid,2021,Jul-Dec,7.0,NaN,NaN,NaN
2,Abia,Fowl typhoid,2025,Jan-Jun,1.0,7.0,7.000000,7.0
3,Abia,Infectious bursal disease (Gumboro disease),2021,Jan-Jun,2.0,NaN,NaN,NaN
4,Abia,Infectious bursal disease (Gumboro disease),2021,Jul-Dec,5.0,2.0,2.000000,2.0
5,Abia,Infectious bursal disease (Gumboro disease),2022,Jan-Jun,2.0,5.0,3.500000,5.0
6,Abia,Infectious bursal disease (Gumboro disease),2022,Jul-Dec,4.0,2.0,3.000000,5.0
9,Abia,Newcastle disease virus (Inf. with),2020,Jul-Dec,1.0,NaN,NaN,NaN
10,Abia,Newcastle disease virus (Inf. with),2021,Jan-Jun,4.0,1.0,1.000000,1.0
11,Abia,Newcastle disease virus (Inf. with),2021,Jul-Dec,9.0,4.0,2.500000,4.0
12,Abia,Newcastle disease virus (Inf. with),2022,Jan-Jun,8.0,9.0,4.666667,9.0



SEMESTER-LEVEL BASELINE HISTORY VALIDATION PASSED


In [3]:
# ============================================================
# CELL 4 — PREPARE BASELINE EVALUATION DATASET
# ============================================================
#
# Purpose:
# Merge the semester-level baseline predictions with the
# observed modelling periods and create a clean evaluation
# dataset.
#
# Each row represents one unique:
#
#     ADM1 × Disease × Year × Semester
#
# Only periods with an observed outbreak count are eligible
# for supervised baseline evaluation.
#
# Baseline predictions remain NaN when the required historical
# information is unavailable.
# ============================================================

# ------------------------------------------------------------
# 1. Select observed target periods
# ------------------------------------------------------------

observed_periods = history_table[
    history_table["target_outbreak_count"].notna()
].copy()

# ------------------------------------------------------------
# 2. Validate uniqueness of the evaluation unit
# ------------------------------------------------------------

evaluation_key = [
    "adm1_key",
    "disease",
    "year",
    "semester_name"
]

duplicate_evaluation_keys = observed_periods.duplicated(
    subset=evaluation_key
).sum()

assert duplicate_evaluation_keys == 0, (
    f"Duplicate observed modelling periods detected: "
    f"{duplicate_evaluation_keys}"
)

# ------------------------------------------------------------
# 3. Sort chronologically
# ------------------------------------------------------------

observed_periods = observed_periods.sort_values(
    [
        "temporal_order",
        "adm1_key",
        "disease"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Check the target distribution
# ------------------------------------------------------------

assert (
    observed_periods["target_outbreak_count"] > 0
).all(), "Non-positive observed outbreak counts detected."

# ------------------------------------------------------------
# 5. Create baseline availability flags
# ------------------------------------------------------------

observed_periods["previous_semester_available"] = (
    observed_periods["baseline_previous_semester"].notna()
)

observed_periods["previous_3_mean_available"] = (
    observed_periods["baseline_previous_3_mean"].notna()
)

observed_periods["previous_3_max_available"] = (
    observed_periods["baseline_previous_3_max"].notna()
)

# ------------------------------------------------------------
# 6. Display evaluation coverage
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE EVALUATION DATASET")
print("=" * 70)

print(
    f"\nObserved target periods: "
    f"{len(observed_periods):,}"
)

print("\nBaseline evaluation coverage:")

baseline_availability = pd.DataFrame({
    "baseline": [
        "Previous semester",
        "Previous 3-semester mean",
        "Previous 3-semester maximum"
    ],
    "available_periods": [
        observed_periods["previous_semester_available"].sum(),
        observed_periods["previous_3_mean_available"].sum(),
        observed_periods["previous_3_max_available"].sum()
    ]
})

baseline_availability["coverage_percent"] = (
    baseline_availability["available_periods"]
    / len(observed_periods)
    * 100
)

display(baseline_availability)

# ------------------------------------------------------------
# 7. Show a sample of the evaluation data
# ------------------------------------------------------------

print("\nSample observed modelling periods:")

display(
    observed_periods[
        [
            "adm1_key",
            "disease",
            "year",
            "semester_name",
            "target_outbreak_count",
            "baseline_previous_semester",
            "baseline_previous_3_mean",
            "baseline_previous_3_max"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 8. Final validation
# ------------------------------------------------------------

assert len(observed_periods) == 445, (
    f"Expected 445 observed periods, found "
    f"{len(observed_periods)}"
)

assert observed_periods["target_outbreak_count"].notna().all()

print(
    "\nBASELINE EVALUATION DATASET VALIDATION PASSED"
)

BASELINE EVALUATION DATASET

Observed target periods: 445

Baseline evaluation coverage:


,baseline,available_periods,coverage_percent
0,Previous semester,310,69.662921
1,Previous 3-semester mean,340,76.404494
2,Previous 3-semester maximum,340,76.404494



Sample observed modelling periods:


,adm1_key,disease,year,semester_name,target_outbreak_count,baseline_previous_semester,baseline_previous_3_mean,baseline_previous_3_max
0,Bayelsa,Newcastle disease virus (Inf. with),2020,Jan-Jun,6.0,NaN,NaN,NaN
1,Borno,Newcastle disease virus (Inf. with),2020,Jan-Jun,3.0,NaN,NaN,NaN
2,Cross River,Fowl typhoid,2020,Jan-Jun,1.0,NaN,NaN,NaN
3,Jigawa,Newcastle disease virus (Inf. with),2020,Jan-Jun,5.0,NaN,NaN,NaN
4,Kaduna,Newcastle disease virus (Inf. with),2020,Jan-Jun,3.0,NaN,NaN,NaN
5,Katsina,Newcastle disease virus (Inf. with),2020,Jan-Jun,1.0,NaN,NaN,NaN
6,Niger,Fowl typhoid,2020,Jan-Jun,1.0,NaN,NaN,NaN
7,Niger,Newcastle disease virus (Inf. with),2020,Jan-Jun,5.0,NaN,NaN,NaN
8,Ondo,Newcastle disease virus (Inf. with),2020,Jan-Jun,2.0,NaN,NaN,NaN
9,Osun,Newcastle disease virus (Inf. with),2020,Jan-Jun,3.0,NaN,NaN,NaN



BASELINE EVALUATION DATASET VALIDATION PASSED


In [4]:
# ============================================================
# CELL 5 — EVALUATE HISTORICAL BASELINE PERFORMANCE
# ============================================================
#
# Purpose:
# Evaluate the three historical baseline strategies using only
# observed outbreak-count periods where each baseline is available.
#
# Metrics:
#   - MAE  : Mean Absolute Error
#   - RMSE : Root Mean Squared Error
#   - MedAE: Median Absolute Error
#
# MAE and RMSE are particularly useful here because the target
# is a non-negative outbreak count and the distribution is
# strongly right-skewed.
#
# We do NOT use MAPE because percentage errors become unstable
# when target values are small. The observed targets include
# many values of 1.
#
# No missing baseline prediction is replaced with zero.
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error
)

# ------------------------------------------------------------
# 1. Define baseline models
# ------------------------------------------------------------

baseline_models = {
    "Previous semester": "baseline_previous_semester",
    "Previous 3-semester mean": "baseline_previous_3_mean",
    "Previous 3-semester maximum": "baseline_previous_3_max"
}

# ------------------------------------------------------------
# 2. Evaluate each baseline independently
# ------------------------------------------------------------

baseline_results = []

for baseline_name, prediction_column in baseline_models.items():

    # Only evaluate periods where both the target and
    # baseline prediction are available.
    evaluation_subset = observed_periods[
        observed_periods[prediction_column].notna()
    ].copy()

    y_true = evaluation_subset["target_outbreak_count"]
    y_pred = evaluation_subset[prediction_column]

    # Calculate absolute errors
    absolute_errors = np.abs(y_true - y_pred)

    # Calculate standard regression metrics
    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    medae = median_absolute_error(y_true, y_pred)

    baseline_results.append({
        "baseline": baseline_name,
        "n_evaluated": len(evaluation_subset),
        "coverage_percent": (
            len(evaluation_subset)
            / len(observed_periods)
            * 100
        ),
        "MAE": mae,
        "RMSE": rmse,
        "Median_AE": medae
    })

# ------------------------------------------------------------
# 3. Create results table
# ------------------------------------------------------------

baseline_results_df = pd.DataFrame(
    baseline_results
)

# Sort by MAE because lower error is better
baseline_results_df = baseline_results_df.sort_values(
    "MAE"
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Display baseline performance
# ------------------------------------------------------------

print("=" * 70)
print("HISTORICAL BASELINE PERFORMANCE")
print("=" * 70)

display(
    baseline_results_df.round(3)
)

# ------------------------------------------------------------
# 5. Identify the strongest baseline
# ------------------------------------------------------------

best_baseline = baseline_results_df.iloc[0]

print("\nBest baseline by MAE:")
print(f"  Model: {best_baseline['baseline']}")
print(f"  MAE: {best_baseline['MAE']:.3f}")
print(f"  RMSE: {best_baseline['RMSE']:.3f}")
print(f"  Median absolute error: {best_baseline['Median_AE']:.3f}")
print(f"  Evaluated periods: {int(best_baseline['n_evaluated'])}")
print(f"  Coverage: {best_baseline['coverage_percent']:.2f}%")

# ------------------------------------------------------------
# 6. Validate metric outputs
# ------------------------------------------------------------

assert len(baseline_results_df) == 3

assert (
    baseline_results_df["n_evaluated"] > 0
).all()

assert (
    baseline_results_df["MAE"] >= 0
).all()

assert (
    baseline_results_df["RMSE"] >= 0
).all()

assert (
    baseline_results_df["Median_AE"] >= 0
).all()

print(
    "\nBASELINE PERFORMANCE EVALUATION PASSED"
)

HISTORICAL BASELINE PERFORMANCE


,baseline,n_evaluated,coverage_percent,MAE,RMSE,Median_AE
0,Previous 3-semester mean,340,76.404,4.883,12.278,2.0
1,Previous semester,310,69.663,4.990,13.055,2.0
2,Previous 3-semester maximum,340,76.404,6.615,17.634,3.0



Best baseline by MAE:
  Model: Previous 3-semester mean
  MAE: 4.883
  RMSE: 12.278
  Median absolute error: 2.000
  Evaluated periods: 340
  Coverage: 76.40%

BASELINE PERFORMANCE EVALUATION PASSED


In [5]:
# ============================================================
# CELL 6 — BASELINE PERFORMANCE BY YEAR
# ============================================================
#
# Purpose:
# Examine whether baseline performance is consistent across
# the available modelling years.
#
# This prevents us from relying only on one overall metric.
# A baseline may perform well overall but fail during particular
# years or periods.
#
# Each baseline is evaluated only where its historical
# prediction is available.
# ============================================================

# ------------------------------------------------------------
# 1. Define baseline models
# ------------------------------------------------------------

baseline_models = {
    "Previous semester": "baseline_previous_semester",
    "Previous 3-semester mean": "baseline_previous_3_mean",
    "Previous 3-semester maximum": "baseline_previous_3_max"
}

# ------------------------------------------------------------
# 2. Calculate yearly performance
# ------------------------------------------------------------

yearly_baseline_results = []

for baseline_name, prediction_column in baseline_models.items():

    # Keep only observations for which this particular
    # baseline has a valid historical prediction.
    baseline_data = observed_periods[
        observed_periods[prediction_column].notna()
    ].copy()

    for year, year_group in baseline_data.groupby("year"):

        y_true = year_group["target_outbreak_count"]
        y_pred = year_group[prediction_column]

        mae = mean_absolute_error(
            y_true,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )

        medae = median_absolute_error(
            y_true,
            y_pred
        )

        yearly_baseline_results.append({
            "year": int(year),
            "baseline": baseline_name,
            "n_evaluated": len(year_group),
            "MAE": mae,
            "RMSE": rmse,
            "Median_AE": medae
        })

# ------------------------------------------------------------
# 3. Create results dataframe
# ------------------------------------------------------------

yearly_baseline_results_df = pd.DataFrame(
    yearly_baseline_results
)

yearly_baseline_results_df = (
    yearly_baseline_results_df
    .sort_values(
        ["year", "MAE", "baseline"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Display detailed yearly results
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE PERFORMANCE BY YEAR")
print("=" * 70)

display(
    yearly_baseline_results_df.round(3)
)

# ------------------------------------------------------------
# 5. Display MAE comparison by year
# ------------------------------------------------------------

yearly_mae_table = (
    yearly_baseline_results_df
    .pivot(
        index="year",
        columns="baseline",
        values="MAE"
    )
)

print("\nMAE BY YEAR:")
display(
    yearly_mae_table.round(3)
)

# ------------------------------------------------------------
# 6. Validate results
# ------------------------------------------------------------

assert not yearly_baseline_results_df.empty

assert (
    yearly_baseline_results_df["MAE"] >= 0
).all()

assert (
    yearly_baseline_results_df["RMSE"] >= 0
).all()

assert (
    yearly_baseline_results_df["Median_AE"] >= 0
).all()

# Every reported year must be within the observed modelling
# period identified earlier.
assert (
    yearly_baseline_results_df["year"].between(
        2020,
        2025
    )
).all()

print(
    "\nYEARLY BASELINE PERFORMANCE VALIDATION PASSED"
)

BASELINE PERFORMANCE BY YEAR


,year,baseline,n_evaluated,MAE,RMSE,Median_AE
0,2020,Previous 3-semester maximum,8,1.500,2.500,0.000
1,2020,Previous 3-semester mean,8,1.500,2.500,0.000
2,2020,Previous semester,8,1.500,2.500,0.000
3,2021,Previous 3-semester maximum,60,5.350,20.145,2.000
4,2021,Previous semester,59,5.424,20.327,2.000
5,2021,Previous 3-semester mean,60,5.564,20.187,2.417
6,2022,Previous 3-semester mean,85,4.073,8.802,1.500
7,2022,Previous semester,78,5.000,13.707,2.000
8,2022,Previous 3-semester maximum,85,6.212,19.311,2.000
9,2023,Previous semester,40,6.025,10.059,3.000



MAE BY YEAR:


baseline,Previous 3-semester maximum,Previous 3-semester mean,Previous semester
year,,,
2020,1.500,1.500,1.500
2021,5.350,5.564,5.424
2022,6.212,4.073,5.000
2023,12.268,8.386,6.025
2024,5.839,3.893,4.535
2025,6.200,4.515,4.744



YEARLY BASELINE PERFORMANCE VALIDATION PASSED


In [6]:
# ============================================================
# CELL 7 — BASELINE PERFORMANCE BY DISEASE
# ============================================================
#
# Purpose:
# Evaluate baseline performance separately for each poultry
# disease category.
#
# This is important because disease frequency is highly
# imbalanced in the WAHIS dataset, with Newcastle disease
# representing a large proportion of observed records.
#
# Each baseline is evaluated only where its historical
# prediction is available.
# ============================================================

# ------------------------------------------------------------
# 1. Define baseline models
# ------------------------------------------------------------

baseline_models = {
    "Previous semester": "baseline_previous_semester",
    "Previous 3-semester mean": "baseline_previous_3_mean",
    "Previous 3-semester maximum": "baseline_previous_3_max"
}

# ------------------------------------------------------------
# 2. Calculate disease-level performance
# ------------------------------------------------------------

disease_baseline_results = []

for baseline_name, prediction_column in baseline_models.items():

    # Keep only observations where the selected baseline
    # is available.
    baseline_data = observed_periods[
        observed_periods[prediction_column].notna()
    ].copy()

    for disease, disease_group in baseline_data.groupby(
        "disease"
    ):

        y_true = disease_group[
            "target_outbreak_count"
        ]

        y_pred = disease_group[
            prediction_column
        ]

        mae = mean_absolute_error(
            y_true,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )

        medae = median_absolute_error(
            y_true,
            y_pred
        )

        disease_baseline_results.append({
            "disease": disease,
            "baseline": baseline_name,
            "n_evaluated": len(disease_group),
            "MAE": mae,
            "RMSE": rmse,
            "Median_AE": medae
        })

# ------------------------------------------------------------
# 3. Create results dataframe
# ------------------------------------------------------------

disease_baseline_results_df = pd.DataFrame(
    disease_baseline_results
)

disease_baseline_results_df = (
    disease_baseline_results_df
    .sort_values(
        ["disease", "MAE", "baseline"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Display detailed disease-level results
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE PERFORMANCE BY DISEASE")
print("=" * 70)

display(
    disease_baseline_results_df.round(3)
)

# ------------------------------------------------------------
# 5. Create disease × baseline MAE comparison
# ------------------------------------------------------------

disease_mae_table = (
    disease_baseline_results_df
    .pivot(
        index="disease",
        columns="baseline",
        values="MAE"
    )
)

print("\nMAE BY DISEASE:")
display(
    disease_mae_table.round(3)
)

# ------------------------------------------------------------
# 6. Identify the best baseline for each disease
# ------------------------------------------------------------

best_baseline_by_disease = (
    disease_baseline_results_df
    .loc[
        disease_baseline_results_df
        .groupby("disease")["MAE"]
        .idxmin()
    ]
    [
        [
            "disease",
            "baseline",
            "n_evaluated",
            "MAE",
            "RMSE",
            "Median_AE"
        ]
    ]
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\nBEST BASELINE BY DISEASE:")
display(
    best_baseline_by_disease.round(3)
)

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

assert not disease_baseline_results_df.empty

assert (
    disease_baseline_results_df["MAE"] >= 0
).all()

assert (
    disease_baseline_results_df["RMSE"] >= 0
).all()

assert (
    disease_baseline_results_df["Median_AE"] >= 0
).all()

print(
    "\nDISEASE-LEVEL BASELINE PERFORMANCE VALIDATION PASSED"
)

BASELINE PERFORMANCE BY DISEASE


,disease,baseline,n_evaluated,MAE,RMSE,Median_AE
0,Avian infectious bronchitis,Previous 3-semester mean,7,4.429,7.435,2.000
1,Avian infectious bronchitis,Previous 3-semester maximum,7,6.143,9.281,2.000
2,Avian infectious bronchitis,Previous semester,5,7.200,10.526,2.000
3,Fowl typhoid,Previous semester,21,1.714,2.690,1.000
4,Fowl typhoid,Previous 3-semester mean,33,2.328,4.553,1.500
5,Fowl typhoid,Previous 3-semester maximum,33,2.606,4.729,2.000
6,Infectious bursal disease (Gumboro disease),Previous 3-semester mean,86,2.855,6.887,1.000
7,Infectious bursal disease (Gumboro disease),Previous semester,73,3.110,7.308,1.000
8,Infectious bursal disease (Gumboro disease),Previous 3-semester maximum,86,3.384,7.769,2.000
9,Newcastle disease virus (Inf. with),Previous semester,211,5.915,15.119,2.000



MAE BY DISEASE:


baseline,Previous 3-semester maximum,Previous 3-semester mean,Previous semester
disease,,,
Avian infectious bronchitis,6.143,4.429,7.200
Fowl typhoid,2.606,2.328,1.714
Infectious bursal disease (Gumboro disease),3.384,2.855,3.110
Newcastle disease virus (Inf. with),8.547,6.107,5.915



BEST BASELINE BY DISEASE:


,disease,baseline,n_evaluated,MAE,RMSE,Median_AE
0,Fowl typhoid,Previous semester,21,1.714,2.690,1.0
1,Infectious bursal disease (Gumboro disease),Previous 3-semester mean,86,2.855,6.887,1.0
2,Avian infectious bronchitis,Previous 3-semester mean,7,4.429,7.435,2.0
3,Newcastle disease virus (Inf. with),Previous semester,211,5.915,15.119,2.0



DISEASE-LEVEL BASELINE PERFORMANCE VALIDATION PASSED


In [7]:
# ============================================================
# CELL 8 — BASELINE ERROR ANALYSIS BY OUTBREAK MAGNITUDE
# ============================================================
#
# Purpose:
# Examine baseline performance across different outbreak-count
# ranges.
#
# The target distribution is strongly right-skewed, so overall
# MAE can hide poor performance on large outbreaks.
#
# Outbreak-count groups:
#   1–3   : low observed counts
#   4–7   : moderate observed counts
#   8–20  : high observed counts
#   >20   : very high observed counts
#
# These thresholds are descriptive groups, not disease-control
# thresholds.
# ============================================================

# ------------------------------------------------------------
# 1. Create outbreak magnitude categories
# ------------------------------------------------------------

def classify_outbreak_magnitude(value):

    if value <= 3:
        return "1–3"

    elif value <= 7:
        return "4–7"

    elif value <= 20:
        return "8–20"

    else:
        return ">20"


magnitude_data = observed_periods.copy()

magnitude_data["outbreak_magnitude"] = (
    magnitude_data["target_outbreak_count"]
    .apply(classify_outbreak_magnitude)
)

# ------------------------------------------------------------
# 2. Define category order
# ------------------------------------------------------------

magnitude_order = [
    "1–3",
    "4–7",
    "8–20",
    ">20"
]

magnitude_data["outbreak_magnitude"] = pd.Categorical(
    magnitude_data["outbreak_magnitude"],
    categories=magnitude_order,
    ordered=True
)

# ------------------------------------------------------------
# 3. Define baseline models
# ------------------------------------------------------------

baseline_models = {
    "Previous semester": "baseline_previous_semester",
    "Previous 3-semester mean": "baseline_previous_3_mean",
    "Previous 3-semester maximum": "baseline_previous_3_max"
}

# ------------------------------------------------------------
# 4. Calculate performance by outbreak magnitude
# ------------------------------------------------------------

magnitude_results = []

for baseline_name, prediction_column in baseline_models.items():

    baseline_data = magnitude_data[
        magnitude_data[prediction_column].notna()
    ].copy()

    for magnitude, magnitude_group in baseline_data.groupby(
        "outbreak_magnitude",
        observed=True
    ):

        y_true = magnitude_group[
            "target_outbreak_count"
        ]

        y_pred = magnitude_group[
            prediction_column
        ]

        absolute_error = np.abs(
            y_true - y_pred
        )

        magnitude_results.append({
            "outbreak_magnitude": magnitude,
            "baseline": baseline_name,
            "n_evaluated": len(magnitude_group),
            "MAE": mean_absolute_error(
                y_true,
                y_pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),
            "Median_AE": median_absolute_error(
                y_true,
                y_pred
            ),
            "Mean_Absolute_Error": (
                absolute_error.mean()
            )
        })

# ------------------------------------------------------------
# 5. Create results dataframe
# ------------------------------------------------------------

magnitude_results_df = pd.DataFrame(
    magnitude_results
)

magnitude_results_df = (
    magnitude_results_df
    .sort_values(
        [
            "outbreak_magnitude",
            "MAE",
            "baseline"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Display detailed results
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE PERFORMANCE BY OUTBREAK MAGNITUDE")
print("=" * 70)

display(
    magnitude_results_df.round(3)
)

# ------------------------------------------------------------
# 7. Display MAE comparison
# ------------------------------------------------------------

magnitude_mae_table = (
    magnitude_results_df
    .pivot(
        index="outbreak_magnitude",
        columns="baseline",
        values="MAE"
    )
)

print("\nMAE BY OUTBREAK MAGNITUDE:")
display(
    magnitude_mae_table.round(3)
)

# ------------------------------------------------------------
# 8. Display number of observed targets in each category
# ------------------------------------------------------------

magnitude_distribution = (
    magnitude_data
    .groupby(
        "outbreak_magnitude",
        observed=True
    )
    .size()
    .reset_index(
        name="observed_target_periods"
    )
)

print("\nOBSERVED OUTBREAK MAGNITUDE DISTRIBUTION:")
display(
    magnitude_distribution
)

# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

assert (
    magnitude_data["outbreak_magnitude"]
    .notna()
    .all()
)

assert (
    magnitude_results_df["MAE"] >= 0
).all()

assert (
    magnitude_results_df["RMSE"] >= 0
).all()

print(
    "\nOUTBREAK-MAGNITUDE BASELINE ANALYSIS PASSED"
)

BASELINE PERFORMANCE BY OUTBREAK MAGNITUDE


,outbreak_magnitude,baseline,n_evaluated,MAE,RMSE,Median_AE,Mean_Absolute_Error
0,1–3,Previous semester,137,1.358,2.234,1.000,1.358
1,1–3,Previous 3-semester mean,157,1.490,2.327,1.000,1.490
2,1–3,Previous 3-semester maximum,157,2.229,3.539,1.000,2.229
3,4–7,Previous semester,79,4.038,6.861,3.000,4.038
4,4–7,Previous 3-semester mean,84,4.210,8.443,2.500,4.210
5,4–7,Previous 3-semester maximum,84,6.024,12.279,3.000,6.024
6,8–20,Previous semester,66,5.121,6.339,5.000,5.121
7,8–20,Previous 3-semester mean,70,5.248,6.150,5.167,5.248
8,8–20,Previous 3-semester maximum,70,5.471,7.112,5.000,5.471
9,>20,Previous 3-semester mean,29,24.316,37.950,15.333,24.316



MAE BY OUTBREAK MAGNITUDE:


baseline,Previous 3-semester maximum,Previous 3-semester mean,Previous semester
outbreak_magnitude,,,
1–3,2.229,1.490,1.358
4–7,6.024,4.210,4.038
8–20,5.471,5.248,5.121
>20,34.828,24.316,25.143



OBSERVED OUTBREAK MAGNITUDE DISTRIBUTION:


,outbreak_magnitude,observed_target_periods
0,1–3,245
1,4–7,97
2,8–20,72
3,>20,31



OUTBREAK-MAGNITUDE BASELINE ANALYSIS PASSED


# Baseline Modelling — Final Conclusion

The baseline analysis established transparent historical benchmarks for
predicting observed poultry disease outbreak counts in Nigeria.

Three historical approaches were evaluated:

1. Previous-semester outbreak count
2. Previous 3-semester mean outbreak count
3. Previous 3-semester maximum outbreak count

The previous 3-semester mean produced the best overall baseline
performance, with:

- MAE: 4.883
- RMSE: 12.278
- Median absolute error: 2.000
- Evaluated periods: 340
- Coverage: 76.40%

However, baseline performance varied across years and diseases.
The previous-semester baseline performed particularly well for several
disease groups, including Fowl typhoid and Newcastle disease.

The outbreak-magnitude analysis showed that baseline performance
deteriorates substantially as outbreak magnitude increases. The
previous-semester baseline performed best for the 1–3, 4–7 and 8–20
outbreak groups, while the previous 3-semester mean performed best for
outbreaks above 20 cases. Even the best baseline for the >20 group had
an MAE of 24.316 and an RMSE of 37.950.

These results establish a meaningful benchmark for subsequent machine
learning models.

## Key Modelling Implication

The historical baselines provide useful information about temporal
persistence in poultry disease outbreaks, but they do not adequately
capture the factors associated with larger outbreak events.

Machine learning models should therefore be evaluated against these
baselines using the same chronological validation principles.

Particular attention should be given to:

- overall prediction error;
- performance across outbreak magnitudes;
- disease-specific performance;
- temporal generalisation; and
- whether climate and other predictors provide predictive value beyond
historical outbreak persistence.

## Important Limitation

Only WAHIS records with observed outbreak counts were used as
supervised targets. Unavailable outbreak observations were not treated
as zero.

The current modelling scenario is retrospective/end-of-semester because
the climate variables represent the same semester as the outbreak
target. These models should therefore not yet be described as true
pre-semester early-warning forecasts.

A true prospective early-warning system will require predictors that are
available before or during the prediction window, such as lagged climate
variables, climate forecasts, or other appropriately time-available
information.

## Transition to Machine Learning

Notebook 07 establishes the baseline benchmark.

The next stage is Notebook 08 — Machine Learning Models, where
chronologically validated machine learning models will be developed and
compared against these historical baselines.